In [2]:
import scanpy as sc
import numpy as np
import gc

# Read data with backed mode to avoid loading everything into memory
orig_path = "vcc_data/arc/perturb_processed.h5ad"
adata = sc.read_h5ad(orig_path)

# Get condition information without loading full data
conditions = adata.obs['condition'].values
ctrl_mask = conditions == 'ctrl'
pert_mask = ~ctrl_mask

# Get unique perturbations
pert_list = np.unique(conditions[pert_mask]).tolist()
print(f"Number of perturbations: {len(pert_list)}")

# Calculate split indices
n_ctrl = np.sum(ctrl_mask)
ctrl_split_idx = int(n_ctrl * 0.5)

n_pert = len(pert_list)
pert_split_idx = int(n_pert * 0.6)

train_list = pert_list[:pert_split_idx]
val_list = pert_list[pert_split_idx:]
print(f"Train perturbations: {len(train_list)}, Val perturbations: {len(val_list)}")

# Create masks for train/val split
ctrl_indices = np.where(ctrl_mask)[0]
train_ctrl_mask = np.zeros(len(conditions), dtype=bool)
val_ctrl_mask = np.zeros(len(conditions), dtype=bool)

train_ctrl_mask[ctrl_indices[:ctrl_split_idx]] = True
val_ctrl_mask[ctrl_indices[ctrl_split_idx:]] = True

# Create perturbation masks
train_pert_mask = np.isin(conditions, train_list)
val_pert_mask = np.isin(conditions, val_list)

# Final train and validation masks
train_mask = train_ctrl_mask | train_pert_mask
val_mask = val_ctrl_mask | val_pert_mask

# Process and save train data


Number of perturbations: 150
Train perturbations: 90, Val perturbations: 60


In [3]:
print("Processing training data...")
adata_train = adata[train_mask]
print(f"Shape of adata train: {adata_train.shape}")


Processing training data...
Shape of adata train: (129796, 18080)


In [6]:

# Save train data and immediately delete from memory
adata_train.write_h5ad("vcc_train/arc/perturb_processed.h5ad")
del adata_train
gc.collect()

3572

In [8]:

# Process and save validation data
print("Processing validation data...")
adata_val = adata[val_mask]
print(f"Shape of adata val: {adata_val.shape}")
adata_val.write_h5ad("vcc_val/arc/perturb_processed.h5ad")
del adata_val
del adata
gc.collect()

print("Train and validation datasets created successfully.")

Processing validation data...
Shape of adata val: (91477, 18080)
Train and validation datasets created successfully.


In [ ]:
(orig_path)
ctrl_adata = adata[adata.obs['condition'] == 'ctrl']
adata = adata[adata.obs['condition'] != 'ctrl']
pert_list =  adata.obs['condition'].unique().tolist()
## splitting ctrl 50-50 train-val
ctrl_train = ctrl_adata[:int(len(ctrl_adata)*0.5)]
ctrl_val = ctrl_adata[int(len(ctrl_adata)*0.5):]
print(len(pert_list))
#splitting 60 -40 train -val
train_list = pert_list[:int(len(pert_list)*0.6)]
val_list = pert_list[int(len(pert_list)*0.6):]
print(len(train_list), len(val_list))
adata_train = adata[adata.obs['condition'].isin(train_list)]
adata_val = adata[adata.obs['condition'].isin(val_list)]
## adding ctrl to train and val
adata_train = adata_train.concat(ctrl_train,)
adata_val = adata_val.concat(ctrl_val)

print(f"Shape of adata train: {adata_train.shape}")
print(f"Shape of adata val: {adata_val.shape}")
adata_train.write_h5ad("/scratch/saigum/PerturbationPredictionUsingGeneNetworks/vcc_train/arc/perturb_processed.h5ad")
adata_val.write_h5ad("/scratch/saigum/PerturbationPredictionUsingGeneNetworks/vcc_val/arc/perturb_processed.h5ad")
print("Train and validation datasets created successfully.")

In [1]:
from gears import PertData
valpert_data = PertData('vcc_val/') # specific saved folder
valpert_data.load(data_path="vcc_val/arc")


Downloading...
100%|██████████| 9.46M/9.46M [00:11<00:00, 841kiB/s] 
Downloading...
100%|██████████| 559k/559k [00:05<00:00, 110kiB/s]  
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['ctrl+SV2A' 'ctrl+WFS1' 'ctrl+SRC' 'ctrl+RAB3B' 'ctrl+SNCA']
Creating pyg object for each cell in the data...
Creating dataset file...
100%|██████████| 56/56 [02:29<00:00,  2.67s/it]
Done!
Saving new dataset pyg object at vcc_val/arc/data_pyg/cell_graphs.pkl
Done!


In [2]:
trainpert_data = PertData('vcc_train/') # specific saved folder
trainpert_data.load(data_path="vcc_train/arc")


Downloading...
100%|██████████| 9.46M/9.46M [00:17<00:00, 536kiB/s] 
Downloading...
100%|██████████| 559k/559k [00:05<00:00, 111kiB/s]  
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['ctrl+ANTXR1' 'ctrl+CAMSAP2' 'ctrl+KDR' 'ctrl+ATP1B1' 'ctrl+PMEL'
 'ctrl+NREP' 'ctrl+DZIP3' 'ctrl+LAD1']
Creating pyg object for each cell in the data...
Creating dataset file...
100%|██████████| 83/83 [04:13<00:00,  3.05s/it]
Done!
Saving new dataset pyg object at vcc_train/arc/data_pyg/cell_graphs.pkl
Done!


In [3]:
import pickle as pk
with open("vcc_val/arc/data_pyg/cell_graphs.pkl","rb") as f:
    cell_graphs = pk.load(f)
print(cell_graphs)

{'ctrl+SHPRH': [Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], pert_idx=[1], de_idx=[1], pert='ctrl+SHPRH'), Data(x=[18080, 1], y=[1, 18080], per

In [27]:
cell_graphs["ctrl+POLB"][0].y[:,:500]

tensor([[ 0.,  0.,  0.,  1.,  0.,  0.,  6.,  5.,  0.,  0.,  0.,  0.,  0.,  8.,
          0.,  0.,  8.,  1.,  2.,  2.,  2.,  1.,  0.,  3.,  0.,  1.,  9.,  0.,
          0.,  1.,  0.,  2.,  4.,  1.,  4.,  0.,  2.,  0.,  1.,  0.,  2.,  0.,
          0.,  0.,  0.,  0.,  0.,  0., 11., 10.,  1.,  0.,  1.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  1.,  2.,  1.,  0.,  0.,  0.,  4.,  1.,  0.,  2.,
          3.,  1.,  0.,  0.,  0.,  6.,  0.,  0.,  7.,  0.,  0.,  0.,  2.,  1.,
          0.,  1.,  1.,  1.,  1.,  7.,  8.,  2.,  0.,  0.,  0.,  1.,  0., 10.,
         29.,  0.,  0.,  0.,  0.,  0.,  0., 11.,  1.,  4.,  2.,  7.,  4.,  0.,
          0.,  0., 17.,  5.,  6.,  0.,  7.,  0.,  6.,  0.,  0., 19.,  3.,  6.,
          0.,  0.,  0.,  4.,  2.,  0., 71.,  7.,  3.,  0.,  2.,  3.,  0.,  0.,
          2.,  1.,  1.,  0.,  1.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., 11.,  0.,
          1.,  0.,  0.,  5.,  0.,  0.,  0.,  1.,  0.

In [32]:
import scanpy as sc
import numpy as np
from cell_load.utils.data_utils import filter_on_target_knockdown
orig_path="vcc_data/adata_Training.h5ad"
data_path="vcc_data/arc/perturb_processed.h5ad"
adata = sc.read_h5ad(orig_path)
sc.pp.log1p(adata)
adata.obs["condition"] = adata.obs["target_gene"]
adata.obs["condition"] = (
    adata.obs["target_gene"].replace("non-targeting","ctrl")
)
adata.var["gene_name"] = adata.var.index.values
print(adata.shape)
# adata = filter_on_target_knockdown(
#     adata=adata,
#     perturbation_column="condition",           # Column in obs containing perturbation info
#     control_label="ctrl",                 # Label for control cells
#     residual_expression=0.30,             # Perturbation-level threshold (30% residual = 70% knockdown)
#     cell_residual_expression=0.50,        # Cell-level threshold (50% residual = 50% knockdown)
#     min_cells=30,                         # Minimum cells per perturbation after filtering
#     layer=None,                           # Use adata.X (or specify a layer)
#     var_gene_name="gene_name"             # Column in var containing gene names
# )   
# print(f"Shape of adata after filtering {adata.shape}")  

adata.obs["condition"]=adata.obs['condition'].cat.rename_categories({cat:f"ctrl+{cat}" for cat in adata.obs.condition.cat.categories if cat != "ctrl"})
adata.obs["cell_type"] = "h1_ESC"
adata.obs["dose_val"] = np.where(adata.obs["condition"] == "ctrl", "1", "1+1")
adata.obs["condition_name"] = adata.obs["cell_type"].astype(str)+"_"+ adata.obs["condition"].astype(str) + "_" + adata.obs["dose_val"].astype(str)


(221273, 18080)


In [ ]:
# computing base statistics
# computing base statistics
cell_means =    np.array(adata.X.mean(axis=1))


In [43]:
cell_means

array([[0.7766858 ],
       [0.48960656],
       [0.9220295 ],
       ...,
       [0.82498276],
       [0.51101035],
       [0.7260371 ]], shape=(221273, 1), dtype=float32)

In [44]:
sc.pp.highly_variable_genes(adata=adata)


In [47]:
adata.var["highly_variable"].sum()

np.int64(1792)

In [53]:
adata.X[:,adata.var["highly_variable"].values]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 270619259 stored elements and shape (221273, 1792)>

In [ ]:
from scipy.sparse import csr_matrix

In [55]:
adata.obsm["X_hvg"] = adata.X[:,adata.var["highly_variable"].values].toarray()

In [56]:
adata

AnnData object with n_obs × n_vars = 221273 × 18080
    obs: 'target_gene', 'guide_id', 'batch', 'condition', 'cell_type', 'dose_val', 'condition_name'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt', 'ribo', 'n_cells', 'gene_name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    obsm: 'X_hvg'

In [57]:
adata.obs["batch"] = 1

In [59]:
adata.write_h5ad("state_.h5ad")

In [65]:
adata.obs.cell_type.unique()

['h1_ESC']
Categories (1, object): ['h1_ESC']

In [ ]:
import pandas as pd
val = ['CHMP3', 'AKT2', 'SHPRH', 'TMSB4X', 'KLF10', 'TARBP2', 'KDM2B', 'SV2A', 'CLDN6', 'TCF3', 'ANTXR1', 'NDUFB6', 'TADA1', 'MED12', 'CAMSAP2', 'IDE', 'PRCP', 'WFS1', 'FOXH1', 'SMARCA4', 'TWF2', 'SAFB', 'POLB', 'TSC22D4', 'ACVR1B', 'PMS1', 'NISCH', 'INSIG1', 'DHCR24', 'MAP3K7', 'TMSB10', 'SMARCA5', 'STAG2', 'ZNF426', 'DNMT1', 'SSBP1']



In [ ]:
# sc.tl.rank_genes_groups(adata, groupby="condition", n_genes=5000, method="wilcoxon")
adata.uns["rank_genes_groups_cov_all"]  = adata.uns["rank_genes_groups"]
adata.write_h5ad(filename=data_path)
print(adata)
print(adata.obs)
